# Demo 1 — Gold Aggregation

This notebook creates dashboard-ready Gold tables from the two Silver tables.

It creates:

- `demo1_crypto_latest_prices_gold`
- `demo1_crypto_market_summary_gold`

The Gold layer combines business-ready metrics from:

- historical daily OHLCV data;
- live streaming price events.

The result is designed for Databricks SQL dashboards and board presentation queries.


## 1. Load shared configuration

In [0]:
%run ../config/00_config


## 2. Import required Spark functions

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


## 3. Confirm both Silver tables exist

In [0]:
required_silver_tables = [
    historical_silver_table,
    streaming_silver_table,
]

missing_silver_tables = [
    table_name
    for table_name in required_silver_tables
    if not spark.catalog.tableExists(table_name)
]

if missing_silver_tables:
    raise RuntimeError(
        f"Missing Silver tables: {missing_silver_tables}"
    )

print("Both Silver tables are available.")


## 4. Read the Silver tables

In [0]:
historical_silver_df = spark.table(
    historical_silver_table
)

streaming_silver_df = spark.table(
    streaming_silver_table
)

print(
    f"Historical Silver rows: "
    f"{historical_silver_df.count()}"
)

print(
    f"Streaming Silver rows: "
    f"{streaming_silver_df.count()}"
)


# Part A — Latest Prices Gold

## 5. Select the latest streaming event per symbol

The latest event is selected using:

- symbol partition;
- event time descending;
- ingestion time descending as a tie-breaker.


In [0]:
latest_event_window = (
    Window
    .partitionBy("symbol")
    .orderBy(
        F.col("event_time").desc(),
        F.col("ingested_at").desc(),
    )
)

latest_streaming_df = (
    streaming_silver_df
    .withColumn(
        "_row_number",
        F.row_number().over(
            latest_event_window
        ),
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)


## 6. Select the latest historical close per symbol

The latest historical daily candle provides the most recent batch close available in the historical dataset.


In [0]:
latest_historical_window = (
    Window
    .partitionBy("symbol")
    .orderBy(
        F.col("open_time").desc(),
        F.col("ingested_at").desc(),
    )
)

latest_historical_df = (
    historical_silver_df
    .withColumn(
        "_row_number",
        F.row_number().over(
            latest_historical_window
        ),
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
    .select(
        "symbol",
        F.col("trade_date").alias(
            "latest_historical_date"
        ),
        F.col("close").alias(
            "latest_historical_close"
        ),
        F.col("price_change_pct").alias(
            "latest_historical_change_pct"
        ),
    )
)


## 7. Build the latest-prices Gold dataset

The two sources are joined by `symbol`.

This comparison shows:

- latest live price;
- latest historical close;
- absolute difference;
- percentage difference;
- latest available 24-hour market statistics.


In [0]:
latest_prices_gold_df = (
    latest_streaming_df.alias("live")
    .join(
        latest_historical_df.alias("historical"),
        on="symbol",
        how="left",
    )
    .withColumn(
        "price_difference",
        F.col("live.price_usd")
        - F.col(
            "historical.latest_historical_close"
        ),
    )
    .withColumn(
        "price_difference_pct",
        F.when(
            F.col(
                "historical.latest_historical_close"
            ) != 0,
            (
                (
                    F.col("live.price_usd")
                    - F.col(
                        "historical.latest_historical_close"
                    )
                )
                / F.col(
                    "historical.latest_historical_close"
                )
            )
            * 100,
        ),
    )
    .select(
        "symbol",
        F.col("live.price_usd").alias(
            "latest_live_price"
        ),
        F.col("live.event_time").alias(
            "latest_live_event_time"
        ),
        "latest_historical_date",
        "latest_historical_close",
        "latest_historical_change_pct",
        "price_difference",
        "price_difference_pct",
        F.col(
            "live.change_pct_24h"
        ).alias("change_pct_24h"),
        F.col(
            "live.high_price_24h"
        ).alias("high_price_24h"),
        F.col(
            "live.low_price_24h"
        ).alias("low_price_24h"),
        F.col(
            "live.volume_24h"
        ).alias("volume_24h"),
        F.current_timestamp().alias(
            "gold_updated_at"
        ),
    )
)

display(
    latest_prices_gold_df
    .orderBy("symbol")
)


## 8. Write the latest-prices Gold table

In [0]:
(
    latest_prices_gold_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        latest_prices_gold_table
    )
)

print(
    f"Latest-prices Gold table written: "
    f"{latest_prices_gold_table}"
)


# Part B — Market Summary Gold

## 9. Aggregate historical performance per symbol

The historical summary calculates:

- first and last date;
- first open;
- latest close;
- minimum low;
- maximum high;
- total volume;
- total trades;
- average daily change;
- bullish, bearish, and neutral day counts.


In [0]:
historical_first_window = (
    Window
    .partitionBy("symbol")
    .orderBy("open_time")
)

historical_last_window = (
    Window
    .partitionBy("symbol")
    .orderBy(
        F.col("open_time").desc()
    )
)

historical_with_rank_df = (
    historical_silver_df
    .withColumn(
        "_first_row",
        F.row_number().over(
            historical_first_window
        ),
    )
    .withColumn(
        "_last_row",
        F.row_number().over(
            historical_last_window
        ),
    )
)

historical_summary_df = (
    historical_with_rank_df
    .groupBy("symbol")
    .agg(
        F.min("trade_date").alias(
            "historical_start_date"
        ),
        F.max("trade_date").alias(
            "historical_end_date"
        ),
        F.max(
            F.when(
                F.col("_first_row") == 1,
                F.col("open"),
            )
        ).alias("first_open_price"),
        F.max(
            F.when(
                F.col("_last_row") == 1,
                F.col("close"),
            )
        ).alias("last_close_price"),
        F.min("low").alias(
            "period_minimum_low"
        ),
        F.max("high").alias(
            "period_maximum_high"
        ),
        F.sum("volume").alias(
            "period_total_volume"
        ),
        F.sum("number_of_trades").alias(
            "period_total_trades"
        ),
        F.avg("price_change_pct").alias(
            "average_daily_change_pct"
        ),
        F.sum(
            F.when(
                F.col("candle_direction")
                == "BULLISH",
                1,
            ).otherwise(0)
        ).alias("bullish_days"),
        F.sum(
            F.when(
                F.col("candle_direction")
                == "BEARISH",
                1,
            ).otherwise(0)
        ).alias("bearish_days"),
        F.sum(
            F.when(
                F.col("candle_direction")
                == "NEUTRAL",
                1,
            ).otherwise(0)
        ).alias("neutral_days"),
    )
    .withColumn(
        "historical_return_pct",
        F.when(
            F.col("first_open_price") != 0,
            (
                (
                    F.col("last_close_price")
                    - F.col("first_open_price")
                )
                / F.col("first_open_price")
            )
            * 100,
        ),
    )
)


## 10. Aggregate streaming activity per symbol

The streaming summary calculates:

- event count;
- first and latest event times;
- minimum, maximum, and average live prices;
- average ingestion delay;
- number of evolved events.


In [0]:
streaming_summary_df = (
    streaming_silver_df
    .groupBy("symbol")
    .agg(
        F.count("*").alias(
            "streaming_event_count"
        ),
        F.min("event_time").alias(
            "first_streaming_event_time"
        ),
        F.max("event_time").alias(
            "latest_streaming_event_time"
        ),
        F.min("price_usd").alias(
            "streaming_minimum_price"
        ),
        F.max("price_usd").alias(
            "streaming_maximum_price"
        ),
        F.avg("price_usd").alias(
            "streaming_average_price"
        ),
        F.avg(
            "ingestion_delay_seconds"
        ).alias(
            "average_ingestion_delay_seconds"
        ),
        F.count(
            "change_pct_24h"
        ).alias(
            "evolved_event_count"
        ),
    )
)


## 11. Build the market-summary Gold dataset

Historical and streaming summaries are combined by symbol.


In [0]:
market_summary_gold_df = (
    historical_summary_df.alias(
        "historical"
    )
    .join(
        streaming_summary_df.alias(
            "streaming"
        ),
        on="symbol",
        how="left",
    )
    .join(
        latest_prices_gold_df
        .select(
            "symbol",
            "latest_live_price",
            "latest_live_event_time",
            "change_pct_24h",
            "high_price_24h",
            "low_price_24h",
            "volume_24h",
        )
        .alias("latest"),
        on="symbol",
        how="left",
    )
    .withColumn(
        "historical_price_range",
        F.col("period_maximum_high")
        - F.col("period_minimum_low"),
    )
    .withColumn(
        "latest_vs_period_high_pct",
        F.when(
            F.col("period_maximum_high") != 0,
            (
                (
                    F.col("latest_live_price")
                    - F.col(
                        "period_maximum_high"
                    )
                )
                / F.col(
                    "period_maximum_high"
                )
            )
            * 100,
        ),
    )
    .withColumn(
        "latest_vs_period_low_pct",
        F.when(
            F.col("period_minimum_low") != 0,
            (
                (
                    F.col("latest_live_price")
                    - F.col(
                        "period_minimum_low"
                    )
                )
                / F.col(
                    "period_minimum_low"
                )
            )
            * 100,
        ),
    )
    .withColumn(
        "gold_updated_at",
        F.current_timestamp(),
    )
)

display(
    market_summary_gold_df
    .orderBy("symbol")
)


## 12. Write the market-summary Gold table

In [0]:
(
    market_summary_gold_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        market_summary_gold_table
    )
)

print(
    f"Market-summary Gold table written: "
    f"{market_summary_gold_table}"
)


## 13. Validate both Gold tables

In [0]:
latest_prices_result_df = spark.table(
    latest_prices_gold_table
)

market_summary_result_df = spark.table(
    market_summary_gold_table
)

latest_prices_count = (
    latest_prices_result_df.count()
)

market_summary_count = (
    market_summary_result_df.count()
)

expected_symbol_count = len(
    historical_symbols
)

print("Gold validation summary")
print("-----------------------")
print(
    f"Latest-prices rows: "
    f"{latest_prices_count}"
)
print(
    f"Market-summary rows: "
    f"{market_summary_count}"
)
print(
    f"Expected symbols: "
    f"{expected_symbol_count}"
)

if latest_prices_count != expected_symbol_count:
    raise RuntimeError(
        "Latest-prices Gold table does not "
        "contain one row per configured symbol."
    )

if market_summary_count != expected_symbol_count:
    raise RuntimeError(
        "Market-summary Gold table does not "
        "contain one row per configured symbol."
    )

latest_prices_duplicate_count = (
    latest_prices_result_df
    .groupBy("symbol")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

market_summary_duplicate_count = (
    market_summary_result_df
    .groupBy("symbol")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

if latest_prices_duplicate_count > 0:
    raise RuntimeError(
        "Latest-prices Gold contains duplicate symbols."
    )

if market_summary_duplicate_count > 0:
    raise RuntimeError(
        "Market-summary Gold contains duplicate symbols."
    )

print("Both Gold tables validated successfully.")


## 14. Dashboard-ready previews

In [0]:
display(
    latest_prices_result_df
    .select(
        "symbol",
        "latest_live_price",
        "latest_historical_close",
        "price_difference_pct",
        "change_pct_24h",
        "latest_live_event_time",
    )
    .orderBy("symbol")
)

display(
    market_summary_result_df
    .select(
        "symbol",
        "historical_return_pct",
        "period_minimum_low",
        "period_maximum_high",
        "latest_live_price",
        "streaming_event_count",
        "average_ingestion_delay_seconds",
    )
    .orderBy("symbol")
)


## Board explanation

> Silver contains clean detailed records. Gold reduces those records into one business-ready row per cryptocurrency. One Gold table focuses on current prices and historical comparison, while the other summarizes historical performance, streaming activity, and operational metrics.

## Next steps

- create `dashboard/dashboard_queries.sql`;
- create the Databricks workflow in `resources/demo1_job.yml`;
- update `README.md` and screenshots.
